In [1]:
import sys
import zarr
import pandas as pd

from pathlib import Path

In [2]:
sys.path.insert(0, str(Path.cwd().parent))

from hooke_tx.data.constants import (
    ASSAY,
    CELL_TYPE,
    EXPERIMENT,
    WELL,
    EMPTY,
    NEG_CONTROL,
    POS_CONTROL,
    GENE_ONLY,
    MOL_ONLY,
    PERTURBATIONS,
    MULTI_PERT,
    GENE_PERT,
    MOL_PERT,
)

### Import data

In [3]:
obs = pd.read_parquet('/rxrx/data/valence/internal_benchmarking/vcds1/drugscreen__trekseq__v1_1/drugscreen__trekseq__v1_1_obs.parquet')
obs.shape

(30766, 21)

In [4]:
var = pd.read_parquet('/rxrx/data/valence/internal_benchmarking/vcds1/drugscreen__trekseq__v1_1/drugscreen__trekseq__v1_1_var.parquet')
var.shape

(60776, 9)

In [5]:
X = zarr.open("/rxrx/data/valence/internal_benchmarking/vcds1/drugscreen__trekseq__v1_1/drugscreen__trekseq__v1_1_features.zarr", mode="r")
X.shape

(30766, 60776)

### Remove redundant duplicated perturbations

In [6]:
# Explode rows according to length of perturbations field
obs_exploded = obs.explode("perturbations").reset_index()
obs_exploded.shape

(47842, 22)

In [7]:
# Add temporary hash column of perturbations column & remove duplicates
obs_exploded['temp_hash'] = obs_exploded['perturbations'].astype(str)
obs_cleaned = obs_exploded.drop_duplicates(subset=[col for col in obs_exploded if col != "perturbations"])
obs_cleaned.shape

(47809, 23)

In [8]:
# Collapse perturbations back into list
obs_final_perts = obs_cleaned.groupby(obs_cleaned["index"])['perturbations'].apply(list).to_frame()
obs_final_perts.shape

(30766, 1)

In [9]:
# Replace in obs
obs["perturbations"] = obs_final_perts["perturbations"].values

### Analyze perturbations column

In [10]:
counts = {
    "empty": 0,
    "negative_control": 0,
    "positive_control": 0,
    "gene": 0,
    "mol": 0,
    "other": 0,
    "gene_mol": 0,
    "gene_gene": 0,
    "mol_mol": 0,
    "nan": 0,
}

e_indices = []

meta_dict = {
    NEG_CONTROL: [],
    POS_CONTROL: [],
    GENE_ONLY: [],
    MOL_ONLY: [],
}

for idx, ps in enumerate(obs["perturbations"]):
    if len(ps) == 0:
        counts["empty"] += 1
    elif len(ps) == 1:
        if not isinstance(ps[0], dict):
            counts["nan"] += 1
        elif ps[0]["type"] == "genetic":
            if ps[0]["usage_class"] == "query":
                counts["gene"] += 1
            elif ps[0]["usage_class"] == "negative_control":
                counts["negative_control"] += 1
                assert obs["is_negative_control"][idx]
            elif ps[0]["usage_class"] == "positive_control":
                counts["positive_control"] += 1
            else:
                counts["other"] += 1
        elif ps[0]["type"] == "compound":
            # assert ps[0]["usage_class"] != "positive_control"
            counts["mol"] += 1
        
        elif ps[0]["type"] is None:
            counts["other"] += 1
            e_indices.append(idx)
        else:
            raise ValueError(f"Unexpected perturbation type: {ps[0]['type']}")

    elif len(ps) == 2:
        if ps[0]["type"] == "genetic" and ps[1]["type"] != "genetic":
            if ps[1]["type"] == "compound":
                counts["gene_mol"] += 1
            else:
                raise ValueError(f"Unexpected perturbation type: {ps[1]['type']}")
                
        elif ps[0]["type"] != "genetic" and ps[1]["type"] == "genetic":
            if ps[0]["type"] == "compound":
                counts["gene_mol"] += 1
            else:
                raise ValueError(f"Unexpected perturbation type: {ps[0]['type']}")

        elif ps[0]["type"] == "genetic" and ps[1]["type"] == "genetic":
            counts["gene_gene"] += 1

        elif ps[0]["type"] == "compound" and ps[1]["type"] == "compound":
            counts["mol_mol"] += 1
        
        else:
            counts["other"] += 1
            raise ValueError(f"Unexpected perturbation type: {ps[0]['type']} and {ps[1]['type']}")
    else:
        raise ValueError(f"Unexpected number of perturbations: {len(ps)}")

for k, v in counts.items():
    print(k, v)

print("overall", sum(counts.values()))

empty 0
negative_control 4351
positive_control 1393
gene 4851
mol 917
other 0
gene_mol 17043
gene_gene 0
mol_mol 0
nan 2211
overall 30766


In [11]:
print("Above counts sum to len(obs):", sum(counts.values()) == len(obs))

Above counts sum to len(obs): True


### Add metadata columns

**Note:** This part of the code currently needs to dataset specific!

In [12]:
gene_pert_keys_and_type = {
    "type": str,
    "gene_symbol": str,
    "ensembl_gene_id": str,
    "concentration": float,
    "effect_class": str,
    "usage_class": str
}

mol_pert_keys_and_type = {
    "type": str,
    "compound_name": str,
    "smiles": str,
    "concentration": float,
    "usage_class": str
}

def trim_perts(perts: list[dict]):
    gene_perts, mol_perts = [], []
    
    for p in perts:
        if p["type"] == "genetic":
            gene_perts.append(p)
        elif p["type"] == "compound":
            mol_perts.append(p)

    # Reorder gene perts based on p["gene_symbol"]
    gene_perts = sorted(gene_perts, key=lambda x: x["gene_symbol"])
    
    # Reorder mol perts based on p["smiles"]
    mol_perts = sorted(mol_perts, key=lambda x: x["smiles"])

    # Trim to keys and types 
    gene_perts = [
        {k: gene_pert_keys_and_type[k](p[k]) for k in gene_pert_keys_and_type}
        for p in gene_perts
    ]
    
    mol_perts = [
        {k: mol_pert_keys_and_type[k](p[k]) for k in mol_pert_keys_and_type}
        for p in mol_perts
    ]

    gene_perts = tuple([
        (p["type"], p["ensembl_gene_id"], p["effect_class"]) 
        for p in gene_perts
    ])

    mol_perts = tuple([
        (p["type"], p["smiles"], str(p["concentration"]))
        for p in mol_perts
    ])

    return gene_perts, mol_perts, gene_perts + mol_perts

In [13]:
assay_types = []
empty_indicator = []
neg_control_indicator = []
pos_control_indicator = []
gene_only_indicator = []
mol_only_indicator = []
multi_pert_indicator = []
gene_perturbations = []
mol_perturbations = []
reordered_perturbations = []

for idx, row in obs.iterrows():
    perts = row[PERTURBATIONS]

    if len(perts) == 0:
        raise ValueError("No perturbations found")

    elif len(perts) == 1:
        if not isinstance(perts[0], dict):
            empty_indicator.append(True)
            neg_control_indicator.append(False)
            pos_control_indicator.append(False)
            gene_only_indicator.append(False)
            mol_only_indicator.append(False)
            multi_pert_indicator.append(False)

            gene_perturbations.append([])
            mol_perturbations.append([])
            reordered_perturbations.append([])
            assay_types.append("bulk_rna_seq")

            continue

        p = perts[0]
        if p["type"] == "compound":
            assert not row["is_negative_control"], "Compound perturbation found is negative control"
            
            empty_indicator.append(False)
            neg_control_indicator.append(p["usage_class"] == "negative_control")
            pos_control_indicator.append(p["usage_class"] == "positive_control")
            gene_only_indicator.append(False)
            mol_only_indicator.append(True)
            multi_pert_indicator.append(False)

            gene_perts, mol_perts, all_perts = trim_perts(perts)

            gene_perturbations.append(gene_perts)
            mol_perturbations.append(mol_perts)
            reordered_perturbations.append(all_perts)
            assay_types.append("bulk_rna_seq")
            
        elif p["type"] == "genetic":
            empty_indicator.append(False)
            neg_control_indicator.append(p["usage_class"] == "negative_control")
            pos_control_indicator.append(p["usage_class"] == "positive_control")
            gene_only_indicator.append(True)
            mol_only_indicator.append(False)
            multi_pert_indicator.append(False)

            gene_perts, mol_perts, all_perts = trim_perts(perts)

            gene_perturbations.append(gene_perts)
            mol_perturbations.append(mol_perts)
            reordered_perturbations.append(all_perts)
            assay_types.append("bulk_rna_seq")
        
        elif p["type"] is None:
            assert not row["is_negative_control"], "Empty perturbation cannot be negative control"
            
            empty_indicator.append(True)
            neg_control_indicator.append(False)
            pos_control_indicator.append(False)
            gene_only_indicator.append(False)
            mol_only_indicator.append(False)
            multi_pert_indicator.append(False)

            gene_perturbations.append([])
            mol_perturbations.append([])
            reordered_perturbations.append([])
            assay_types.append("bulk_rna_seq")
        
        else:
            raise ValueError(f"Unexpected perturbation type: {p['type']}")  

    elif len(perts) == 2:
        gene_perts, mol_perts, all_perts = trim_perts(perts)

        gene_perturbations.append(gene_perts)
        mol_perturbations.append(mol_perts)
        reordered_perturbations.append(all_perts)
        assay_types.append("bulk_rna_seq")
        
        p1, p2 = perts[0], perts[1]

        if p1["type"] == "genetic" and p2["type"] != "genetic":
            assert p2["type"] == "compound", "Expecting drugscreen type perturbation"

            if p1["usage_class"] in ["query", "negative_control", "positive_control"]:
                empty_indicator.append(False)
                neg_control_indicator.append(False)
                pos_control_indicator.append(False)
                gene_only_indicator.append(False)
                mol_only_indicator.append(False)
                multi_pert_indicator.append(True)

            else:
                raise ValueError(f"Unexpected perturbation type: {p1['type']}")

        elif p1["type"] != "genetic" and p2["type"] == "genetic":
            assert p1["type"] == "compound", "Expecting drugscreen type perturbation"

            if p2["usage_class"] in ["query", "negative_control", "positive_control"]:
                empty_indicator.append(False)
                neg_control_indicator.append(False)
                pos_control_indicator.append(False)
                gene_only_indicator.append(False)
                mol_only_indicator.append(False)
                multi_pert_indicator.append(True)

            else:
                raise ValueError(f"Unexpected perturbation type: {p2['type']}")

        elif p1["type"] == "genetic" and p2["type"] == "genetic":
            raise ValueError("Double genetic perturbations should not occur")

        elif p1["type"] == "compound" and p2["type"] == "compound":
            empty_indicator.append(False)
            neg_control_indicator.append(False)
            pos_control_indicator.append(False)
            gene_only_indicator.append(False)
            mol_only_indicator.append(True)
            multi_pert_indicator.append(True)

        else:
            raise ValueError(f"Unexpected perturbation combination: {p1['type']} and {p2['type']}")
    else:
        raise ValueError(f"Unexpected number of perturbations: {len(perts)}")

In [ ]:
# We only keep some columns (that are needed for training) and add the newly created metadata columns
# We might want to keep the old `perturbtations`` column as `raw_perturbations` because it's required by vcb
obs_extended = obs.copy()

# Reduce to useful columns for training
obs_extended = obs_extended[[CELL_TYPE, EXPERIMENT, WELL]]

# Add columns to obs_extended
obs_extended[ASSAY] = assay_types
obs_extended[PERTURBATIONS] = reordered_perturbations
obs_extended[EMPTY] = empty_indicator
obs_extended[NEG_CONTROL] = neg_control_indicator
obs_extended[POS_CONTROL] = pos_control_indicator
obs_extended[GENE_ONLY] = gene_only_indicator
obs_extended[MOL_ONLY] = mol_only_indicator
obs_extended[MULTI_PERT] = multi_pert_indicator
obs_extended[GENE_PERT] = gene_perturbations
obs_extended[MOL_PERT] = mol_perturbations

obs_extended.shape

(30766, 13)

In [15]:
obs_extended.head(5)

,cell_type,experiment_label,well_address,assay_type,perturbations,empty,negative_control,positive_control,gene_only,mol_only,multi_pert,gene_pert,mol_pert
0,HUVEC,250304-trek-1082-huvec-ipg-SMG6-rescue_p1-a,B02,bulk_rna_seq,"((genetic, ENSG00000070366, KO), (compound, c1...",False,False,False,False,False,True,"((genetic, ENSG00000070366, KO),)","((compound, c1cc(CNc2nc(-c3ccncc3)nc3[nH]ncc23..."
1,HUVEC,250304-trek-1082-huvec-ipg-SMG6-rescue_p1-a,B03,bulk_rna_seq,"((genetic, ENSG00000070366, KO), (compound, c1...",False,False,False,False,False,True,"((genetic, ENSG00000070366, KO),)","((compound, c1cc(CNc2nc(-c3ccncc3)nc3[nH]ncc23..."
2,HUVEC,250304-trek-1082-huvec-ipg-SMG6-rescue_p1-a,B04,bulk_rna_seq,"((genetic, ENSG00000070366, KO), (compound, O=...",False,False,False,False,False,True,"((genetic, ENSG00000070366, KO),)","((compound, O=C(NCCSc1ccccc1)N1CC=C(c2c[nH]c3n..."
3,HUVEC,250304-trek-1082-huvec-ipg-SMG6-rescue_p1-a,B05,bulk_rna_seq,"((genetic, ENSG00000134256, KO),)",False,True,False,True,False,False,"((genetic, ENSG00000134256, KO),)",()
4,HUVEC,250304-trek-1082-huvec-ipg-SMG6-rescue_p1-a,B06,bulk_rna_seq,"((genetic, ENSG00000070366, KO),)",False,False,False,True,False,False,"((genetic, ENSG00000070366, KO),)",()


In [16]:
# obs_extended.to_parquet("/rxrx/scratch/hooke_tx/trek/drugscreen__trekseq__v1_1/obs.parquet")